# Train Final BM25+Model1 Fusion on stackoverflow_all (Optimal Config)

This notebook takes the best parameters found during tuning, creates one final BM25+Model1 descriptor, trains on `train`, evaluates on `dev1`, and prints the final report.

## 1) Setup

- Uses unified collection: `stackoverflow_all`
- Uses tuning outputs from notebook 05:
  - `bm25tune_stackoverflow_all.tsv`
  - `model1tune_stackoverflow_all.tsv`

In [ ]:
import csv
import json
import os
import re
import subprocess
from pathlib import Path

REPO_ROOT = Path('/tempory/the_three_potatoes/ri_project/workspaces/amelie/FlexNeuART-IR-TTP')
SCRIPTS_DIR = REPO_ROOT / 'demo' / 'flexneuart_scripts'
COLLECT_ROOT = (REPO_ROOT / 'demo' / 'collections').resolve()
COLLECT_NAME = 'stackoverflow_all'

TRAIN_PART = 'train'
TEST_PART = 'test' # evaluation on text
TRAIN_CAND_QTY = 15

# Candidate depth to filter in tuning/result extraction scripts
TOP_K = '250'

# Metric used to pick best configuration from tuning tables
SELECT_METRIC_COL = 'MAP'

BM25_TUNE_TSV = SCRIPTS_DIR / 'bm25tune_stackoverflow_all.tsv'
MODEL1_TUNE_TSV = SCRIPTS_DIR / 'model1tune_stackoverflow_all.tsv'

# Output descriptor locations inside the collection
collect_dir = COLLECT_ROOT / COLLECT_NAME
desc_root = collect_dir / 'exper_desc.best'
extractor_dir = desc_root / 'extractors'
desc_root.mkdir(parents=True, exist_ok=True)
extractor_dir.mkdir(parents=True, exist_ok=True)

os.environ['COLLECT_ROOT'] = str(COLLECT_ROOT)

print('REPO_ROOT        =', REPO_ROOT)
print('SCRIPTS_DIR      =', SCRIPTS_DIR)
print('COLLECT_ROOT     =', COLLECT_ROOT)
print('COLLECT_NAME     =', COLLECT_NAME)
print('TRAIN_PART       =', TRAIN_PART)
print('TEST_PART        =', TEST_PART)
print('TRAIN_CAND_QTY   =', TRAIN_CAND_QTY)
print('TOP_K            =', TOP_K)
print('SELECT_METRIC_COL=', SELECT_METRIC_COL)
print('BM25_TUNE_TSV    =', BM25_TUNE_TSV)
print('MODEL1_TUNE_TSV  =', MODEL1_TUNE_TSV)

## 2) Validate prerequisites

In [ ]:
required_paths = [
    BM25_TUNE_TSV,
    MODEL1_TUNE_TSV,
    collect_dir / 'input_data' / TRAIN_PART / 'QuestionFields.jsonl',
    collect_dir / 'input_data' / TRAIN_PART / 'AnswerFields.jsonl',
    collect_dir / 'input_data' / TRAIN_PART / 'qrels.txt',
    collect_dir / 'input_data' / TEST_PART / 'QuestionFields.jsonl',
    collect_dir / 'input_data' / TEST_PART / 'AnswerFields.jsonl',
    collect_dir / 'input_data' / TEST_PART / 'qrels.txt',
    collect_dir / 'lucene_index',
    collect_dir / 'forward_index' / 'text',
    collect_dir / 'derived_data' / 'giza' / 'text' / 'output.t1.5.bin'
]

for p in required_paths:
    if not p.exists():
        raise FileNotFoundError(f'Missing prerequisite: {p}')

print('All prerequisites found.')

## 3) Pick the best params from tuning TSVs

- BM25 params from `bm25tune_stackoverflow_all.tsv`
- Model1 params (`lambda`, `probSelfTran`, optional `minTranProb`) from `model1tune_stackoverflow_all.tsv`

In [ ]:
def load_best_row(tsv_path: Path, metric_col: str, top_k: str):
    with tsv_path.open('r', encoding='utf-8') as f:
        rows = list(csv.DictReader(f, delimiter='\t'))

    if not rows:
        raise RuntimeError(f'No rows in TSV: {tsv_path}')

    filt = [r for r in rows if r.get('top_k') == str(top_k)]
    if not filt:
        raise RuntimeError(f'No rows with top_k={top_k} in {tsv_path}')

    missing = [r for r in filt if metric_col not in r or r[metric_col] in (None, '')]
    if missing:
        raise RuntimeError(f'Missing metric column {metric_col} in some rows of {tsv_path}')

    best = max(filt, key=lambda r: float(r[metric_col]))
    return best

def parse_bm25_from_exper_subdir(exper_subdir: str):
    m = re.search(r'k1=([0-9.]+)_b=([0-9.]+)', exper_subdir)
    if not m:
        raise RuntimeError(f'Cannot parse BM25 params from exper_subdir: {exper_subdir}')
    return float(m.group(1)), float(m.group(2))

def parse_model1_from_exper_subdir(exper_subdir: str):
    m_lambda = re.search(r'lambda=([0-9.]+)', exper_subdir)
    m_prob = re.search(r'probSelfTran=([0-9.]+)', exper_subdir)
    m_min = re.search(r'minTranProb=([0-9.eE+-]+)', exper_subdir)

    if not m_lambda or not m_prob:
        raise RuntimeError(f'Cannot parse Model1 params from exper_subdir: {exper_subdir}')

    lamb = float(m_lambda.group(1))
    prob_self = float(m_prob.group(1))
    min_prob = float(m_min.group(1)) if m_min else 2.5e-3

    return lamb, prob_self, min_prob

bm25_best = load_best_row(BM25_TUNE_TSV, SELECT_METRIC_COL, TOP_K)
model1_best = load_best_row(MODEL1_TUNE_TSV, SELECT_METRIC_COL, TOP_K)

BM25_K1, BM25_B = parse_bm25_from_exper_subdir(bm25_best['exper_subdir'])
MODEL1_LAMBDA, MODEL1_PROB_SELF_TRAN, MODEL1_MIN_PROB = parse_model1_from_exper_subdir(model1_best['exper_subdir'])

print('Best BM25 row:')
print(bm25_best)
print('')
print('Best Model1 row:')
print(model1_best)
print('')
print('Selected params:')
print('BM25_K1 =', BM25_K1)
print('BM25_B =', BM25_B)
print('MODEL1_LAMBDA =', MODEL1_LAMBDA)
print('MODEL1_PROB_SELF_TRAN =', MODEL1_PROB_SELF_TRAN)
print('MODEL1_MIN_PROB =', MODEL1_MIN_PROB)

## 4) Build one final descriptor JSON (best config only)

In [ ]:
fid = (
    f'bm25=text+model1=text'
    f'+k1={BM25_K1:g}+b={BM25_B:g}'
    f'+lambda={MODEL1_LAMBDA:g}+probSelfTran={MODEL1_PROB_SELF_TRAN:g}'
)

extractor_json_rel = Path('exper_desc.best') / 'extractors' / f'{fid}.json'
extractor_json_abs = collect_dir / extractor_json_rel

extractor_json = [
    {
        'type': 'Model1Similarity',
        'params': {
            'queryFieldName': 'text',
            'indexFieldName': 'text',
            'gizaIterQty': '5',
            'probSelfTran': MODEL1_PROB_SELF_TRAN,
            'lambda': MODEL1_LAMBDA,
            'minModel1Prob': MODEL1_MIN_PROB
        }
    },
    {
        'type': 'TFIDFSimilarity',
        'params': {
            'queryFieldName': 'text',
            'indexFieldName': 'text',
            'similType': 'bm25',
            'k1': BM25_K1,
            'b': BM25_B
        }
    }
]

with extractor_json_abs.open('w', encoding='utf-8') as f:
    json.dump(extractor_json, f, indent=2)

final_desc_rel = Path('exper_desc.best') / 'bm25_model1_optimal.json'
final_desc_abs = collect_dir / final_desc_rel

final_desc = [
    {
        'experSubdir': 'feat_exper/bm25_model1_optimal',
        'extrTypeFinal': str(extractor_json_rel).replace('\\', '/'),
        'testOnly': 0
    }
]

with final_desc_abs.open('w', encoding='utf-8') as f:
    json.dump(final_desc, f, indent=2)

print('Created extractor JSON:', extractor_json_abs)
print('Created experiment descriptor:', final_desc_abs)

## 5) Train + evaluate final optimal fusion

This runs training on `train` and evaluation on `dev1` using the single optimal descriptor.

In [ ]:
cmd = [
    'bash', './exper/run_experiments.sh',
    COLLECT_NAME,
    str(final_desc_rel).replace('\\', '/'),
    '-test_part', TEST_PART,
    '-train_part', TRAIN_PART,
    '-train_cand_qty', str(TRAIN_CAND_QTY),
    '-model1_subdir', 'giza'
]

print('Running:', ' '.join(cmd))
res = subprocess.run(cmd, cwd=SCRIPTS_DIR, env=os.environ.copy(), text=True, capture_output=True)
print(res.stdout)
if res.returncode != 0:
    print(res.stderr)
    raise RuntimeError(f'Final BM25+Model1 experiment failed: {res.returncode}')

## 6) Show final report

In [ ]:
report_file = collect_dir / 'results' / TEST_PART / 'feat_exper' / 'bm25_model1_optimal' / 'rep' / 'out_100.rep'

if not report_file.exists():
    raise FileNotFoundError(f'Report not found: {report_file}')

print('Report file:', report_file)
print('')
print(report_file.read_text(encoding='utf-8'))

In [ ]:
summary_tsv = 'bm25_model1_optimal_stackoverflow_all.tsv'
cmd = [
    'bash', './report/get_exper_results.sh',
    COLLECT_NAME,
    str(final_desc_rel).replace('\\', '/'),
    summary_tsv,
    '-test_part', TEST_PART,
    '-flt_cand_qty', TOP_K,
    '-print_best_metr', 'map'
]

print('Running:', ' '.join(cmd))
res = subprocess.run(cmd, cwd=SCRIPTS_DIR, env=os.environ.copy(), text=True, capture_output=True)
print(res.stdout)
if res.returncode != 0:
    print(res.stderr)
    raise RuntimeError(f'Final summary extraction failed: {res.returncode}')

summary_path = SCRIPTS_DIR / summary_tsv
print('Summary TSV:', summary_path)
print(summary_path.read_text(encoding='utf-8'))

## Notes

- If you want to select best config by another metric, change `SELECT_METRIC_COL` (for example to `NDCG@20` or `P@20`).
- This notebook does not re-tune; it reuses tuning outputs and trains one final model using the chosen best parameters.

## 7) Export BM25+Model1 sparse vectors for NMSLIB

This section exports interleaved sparse vectors that approximate BM25+Model1 with inner product search.

- Document vectors are exported with learned fusion weights (`-model_file`).
- Query vectors are exported separately (`-query_file_pref`) for evaluation/benchmarking.

Output files:
- `derived_data/nmslib/bm25_model1_interleaved/docs_export.data`
- `derived_data/nmslib/bm25_model1_interleaved/queries_test_export.data`

In [ ]:
from typing import Iterable

NMSLIB_SUBDIR = Path('nmslib') / 'bm25_model1_interleaved'
docs_export_rel = NMSLIB_SUBDIR / 'docs_export.data'
queries_export_rel = NMSLIB_SUBDIR / f'queries_{TEST_PART}_export.data'

nmslib_out_dir = collect_dir / 'derived_data' / NMSLIB_SUBDIR
nmslib_out_dir.mkdir(parents=True, exist_ok=True)


def looks_like_ranklib_linear_model(path: Path) -> bool:
    try:
        with path.open('r', encoding='utf-8', errors='ignore') as f:
            for line in f:
                line = line.strip()
                if not line or line.startswith('#'):
                    continue
                tokens = line.split()
                if not tokens:
                    continue
                # Expect tokens like: 1:0.123 2:-0.56 ...
                return all(':' in t and t.split(':', 1)[0].isdigit() for t in tokens)
    except Exception:
        return False
    return False


def find_fusion_model_file(root: Path) -> Path:
    patterns = ['*.model', '*.txt', '*.weights']
    cand = []
    for pat in patterns:
        cand.extend(root.rglob(pat))

    filtered = [
        p for p in cand
        if 'feat_exper' in str(p).replace('\\\\', '/')
        and 'bm25_model1_optimal' in str(p).replace('\\\\', '/')
        and p.is_file()
        and looks_like_ranklib_linear_model(p)
    ]

    if not filtered:
        raise FileNotFoundError(
            'Could not auto-locate a RankLib linear model for bm25_model1_optimal. '
            'Please inspect collect_dir/results and set model_file_rel manually.'
        )

    # Pick latest modified candidate
    filtered.sort(key=lambda p: p.stat().st_mtime, reverse=True)
    return filtered[0]


model_file_abs = find_fusion_model_file(collect_dir / 'results')
model_file_rel = model_file_abs.relative_to(collect_dir)

query_file_pref = f'{TEST_PART}/QuestionFields'

print('NMSLIB output directory  =', nmslib_out_dir)
print('Using model file         =', model_file_rel)
print('Extractor JSON           =', extractor_json_rel)
print('Query file prefix        =', query_file_pref)
print('Docs export rel path     =', docs_export_rel)
print('Queries export rel path  =', queries_export_rel)

In [ ]:
def run_export(cmd):
    print('Running:', ' '.join(str(x) for x in cmd))
    res = subprocess.run(
        cmd,
        cwd=SCRIPTS_DIR,
        env=os.environ.copy(),
        text=True,
        capture_output=True
    )
    print(res.stdout)
    if res.returncode != 0:
        print(res.stderr)
        raise RuntimeError(f'Export failed with code {res.returncode}')


cmd_docs = [
    'bash', './export_nmslib/export_nmslib_sparse.sh',
    COLLECT_NAME,
    str(extractor_json_rel).replace('\\', '/'),
    str(docs_export_rel).replace('\\', '/'),
    '-model_file',
    str(model_file_rel).replace('\\', '/')
]

cmd_queries = [
    'bash', './export_nmslib/export_nmslib_sparse.sh',
    COLLECT_NAME,
    str(extractor_json_rel).replace('\\', '/'),
    str(queries_export_rel).replace('\\', '/'),
    '-query_file_pref',
    query_file_pref
]

run_export(cmd_docs)
run_export(cmd_queries)


docs_export_abs = collect_dir / 'derived_data' / docs_export_rel
queries_export_abs = collect_dir / 'derived_data' / queries_export_rel

if not docs_export_abs.exists():
    raise FileNotFoundError(f'Docs export missing: {docs_export_abs}')
if not queries_export_abs.exists():
    raise FileNotFoundError(f'Queries export missing: {queries_export_abs}')

print('Created docs export   :', docs_export_abs)
print('Created queries export:', queries_export_abs)
print('Docs file size (bytes):', docs_export_abs.stat().st_size)
print('Qrys file size (bytes):', queries_export_abs.stat().st_size)

## 8) Evaluation implications

Training does **not** change: you still tune/train BM25+Model1 fusion exactly as above.

What changes is candidate retrieval mode:

- Original exact pipeline: score docs directly with feature extractors.
- NMSLIB pipeline: precompute/index doc vectors and score by inner product with query vectors.

So evaluation can stay the same in metric terms (`MAP`, `NDCG@k`, etc.), but you should expect small differences in candidate sets/scores due to approximation and ANN settings. A common practice is to compare:

1. Exact BM25+Model1 retrieval metrics.
2. NMSLIB retrieval metrics (same qrels, same eval scripts).

If the gap is acceptable, use NMSLIB for faster retrieval.